In [1]:
import requests
import pandas as pd
import time
import os

In [2]:
BASE_URL = "https://datacatalogapi.worldbank.org/dexapps/fone/api/apiservice"
DATASET_ID = "DS00005"
RESOURCE_ID = "RS00005"
SAVE_PATH = "../data/raw/contracts_raw.csv"

In [3]:
params = {
    "datasetId": DATASET_ID,
    "resourceId": RESOURCE_ID,
    "limit": 5,
    "skip": 0
}

response = requests.get(BASE_URL, params=params)
print("Status code:", response.status_code)
print(response.json())

Status code: 200
{'count': 273504, 'data': [{'as_of_date': '21-Apr-2026', 'borrower_contract_reference_number': 'UDP PPG-QCBS-2016-4-12', 'borrower_country': 'Kyrgyz Republic', 'borrower_country_code': 'KG', 'contract_description': 'Technical Supervision of construction in pilot Schools and Kindergartens', 'contract_signing_date': '19-May-2022', 'fiscal_year': 2022, 'procurement_category': 'Consultant Services', 'procurement_method': 'Quality And Cost-Based Selection', 'project_global_practice': 'Public Admin;Energy & Extractives;Transportation;Water/Sanit/Waste', 'project_id': 'P151416', 'project_name': 'Urban Development Project', 'region': 'Europe and Central Asia', 'review_type': 'Prior', 'supplier': 'M/S. AIRES INGEGNERIA', 'supplier_contract_amount_usd': 161538.89, 'supplier_country': 'Italy', 'supplier_country_code': 'IT', 'supplier_id': '316882', 'wb_contract_number': '1543858'}, {'as_of_date': '21-Apr-2026', 'borrower_contract_reference_number': 'UDP PPG-QCBS-2016-4-12', 'borr

In [4]:
all_data = []
limit = 1000
skip = 0

while True:
    params = {
        "datasetId": DATASET_ID,
        "resourceId": RESOURCE_ID,
        "limit": limit,
        "skip": skip
    }
    
    response = requests.get(BASE_URL, params=params)
    batch = response.json().get("data", [])
    
    if not batch:  # stop when no more data
        break
    
    all_data.extend(batch)
    skip += limit
    print(f"Pulled {len(all_data)} rows so far...")
    time.sleep(0.5)  # be polite to the API

print("Done! Total rows:", len(all_data))

Pulled 1000 rows so far...
Pulled 2000 rows so far...
Pulled 3000 rows so far...
Pulled 4000 rows so far...
Pulled 5000 rows so far...
Pulled 6000 rows so far...
Pulled 7000 rows so far...
Pulled 8000 rows so far...
Pulled 9000 rows so far...
Pulled 10000 rows so far...
Pulled 11000 rows so far...
Pulled 12000 rows so far...
Pulled 13000 rows so far...
Pulled 14000 rows so far...
Pulled 15000 rows so far...
Pulled 16000 rows so far...
Pulled 17000 rows so far...
Pulled 18000 rows so far...
Pulled 19000 rows so far...
Pulled 20000 rows so far...
Pulled 21000 rows so far...
Pulled 22000 rows so far...
Pulled 23000 rows so far...
Pulled 24000 rows so far...
Pulled 25000 rows so far...
Pulled 26000 rows so far...
Pulled 27000 rows so far...
Pulled 28000 rows so far...
Pulled 29000 rows so far...
Pulled 30000 rows so far...
Pulled 31000 rows so far...
Pulled 32000 rows so far...
Pulled 33000 rows so far...
Pulled 34000 rows so far...
Pulled 35000 rows so far...
Pulled 36000 rows so far...
P

In [6]:
df_full = pd.DataFrame(all_data)

os.makedirs("../data/raw/", exist_ok=True)
df_full.to_csv(SAVE_PATH, index=False)

print("Saved to", SAVE_PATH)
print(df_full.shape)

Saved to ../data/raw/contracts_raw.csv
(273504, 20)


In [12]:
SAVE_PATH_LOANS = "../data/raw/loans_raw.csv"

all_loans = []
skip = 0

while True:
    params = {
        "datasetId": "DS00001",
        "resourceId": "RS00001",
        "limit": 1000,
        "skip": skip
    }
    
    response = requests.get("https://datacatalogapi.worldbank.org/dexapps/fone/api/apiservice", params=params)
    batch = response.json().get("data", [])
    
    if not batch:
        break
    
    all_loans.extend(batch)
    skip += 1000
    print(f"Pulled {len(all_loans)} rows so far...")
    time.sleep(0.5)

df_loans = pd.DataFrame(all_loans)
df_loans.to_csv(SAVE_PATH_LOANS, index=False)
print("Done! Shape:", df_loans.shape)

Pulled 1000 rows so far...
Pulled 2000 rows so far...
Pulled 3000 rows so far...
Pulled 4000 rows so far...
Pulled 5000 rows so far...
Pulled 6000 rows so far...
Pulled 7000 rows so far...
Pulled 8000 rows so far...
Pulled 9000 rows so far...
Pulled 10000 rows so far...
Pulled 11000 rows so far...
Pulled 11236 rows so far...
Done! Shape: (11236, 30)


In [8]:
SAVE_PATH_COUNTRIES = "../data/raw/countries_raw.csv"

country_url = "https://api.worldbank.org/v2/country?format=json&per_page=300"

response = requests.get(country_url)
raw = response.json()

df_countries = pd.DataFrame(raw[1])
df_countries.to_csv(SAVE_PATH_COUNTRIES, index=False)
print("Done! Shape:", df_countries.shape)

Done! Shape: (296, 10)


In [10]:
SAVE_PATH_EDUCATION = "../data/raw/education_raw.csv"

indicators = [
    "SE.PRM.ENRR",    # Primary school enrollment %
    "SE.SEC.ENRR",    # Secondary school enrollment %
    "SE.ADT.LITR.ZS"  # Adult literacy rate %
]

all_edu = []

for indicator in indicators:
    url = f"https://api.worldbank.org/v2/country/all/indicator/{indicator}?format=json&per_page=1000&mrv=10"
    response = requests.get(url)
    data = response.json()
    
    if len(data) > 1:
        batch = pd.DataFrame(data[1])
        batch["indicator_code"] = indicator
        all_edu.append(batch)
        print(f" {indicator} pulled")
    time.sleep(0.5)

df_education = pd.concat(all_edu, ignore_index=True)
df_education.to_csv(SAVE_PATH_EDUCATION, index=False)
print("Done! Shape:", df_education.shape)

 SE.PRM.ENRR pulled
 SE.SEC.ENRR pulled
 SE.ADT.LITR.ZS pulled
Done! Shape: (3000, 9)
